In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import polars as pl
import numpy as np

from pathlib import Path


### Volume Data Collection

In [37]:
from datetime import date as Date, timedelta

BASE = Path("../data/prices")
date = "2025-02-02"

target_ticker = ["GS"]
predictor_tickers = ["MS", "SPY", "IVV", "XLF"]


def date_range(start: str, end: str):
    cur = Date.fromisoformat(start)
    last = Date.fromisoformat(end)

    while cur <= last:
        yield cur.isoformat()
        cur += timedelta(days=1)


def load_ticker_daily_files(ticker: str, start: str, end: str) -> pd.DataFrame:
    dfs = []

    for d in date_range(start, end):
        path = BASE / f"{ticker}_1s_{d}.parquet"

        if not path.exists():
            continue

        df = (
            pl.scan_parquet(path)
            .select([
                pl.col("ts"),
                pl.col("volume").alias(f"{ticker}_volume"),
            ])
            .collect()
        )

        dfs.append(df)

    if dfs:
        return pl.concat(dfs).sort("ts")

    return pl.DataFrame({"ts": [], f"{ticker}_volume": []})

In [43]:
from functools import reduce

start = "2025-01-01"
end = "2025-06-20"

all_tickers = target_ticker + predictor_tickers

dfs = [
    load_ticker_daily_files(tck, start, end)
    .select(["ts", f"{tck}_volume"])
    .unique(subset=["ts"], keep="first")
    .sort("ts")
    for tck in all_tickers
]

# Best default: full join, keep only one ts column
df_volume = reduce(
    lambda left, right: left.join(
        right,
        on="ts",
        how="full",
        coalesce=True,
    ),
    dfs
).sort("ts")

# Fill missing volume with 0
volume_cols = [c for c in df_volume.columns if c.endswith("_volume")]

df_volume = df_volume.with_columns([
    pl.col(c).fill_null(0).alias(c)
    for c in volume_cols
])

df_volume.head(30).sum()

ts,GS_volume,MS_volume,SPY_volume,IVV_volume,XLF_volume
"datetime[ns, US/Eastern]",f64,f64,f64,f64,f64
null,26119.0,497.0,1.126182e6,73315.0,240736.0


In [45]:
# bucket the volume into 30 second intervals
BUCKET_SIZE = 30 # 30 seconds
symbols = target_ticker + predictor_tickers
volume_cols = [f"{sym}_volume" for sym in symbols]

df_volume_bucket = (
    df_volume
    .sort("ts")
    .group_by_dynamic(
        index_column="ts",
        every=f"{BUCKET_SIZE}s",
        period=f"{BUCKET_SIZE}s",
        closed="left"
    )
    .agg([
        pl.col(c).sum().alias(c)
        for c in volume_cols
    ])
)

# df_volume_bucket = df_volume_bucket.to_pandas()
df_volume_bucket.head()


ts,GS_volume,MS_volume,SPY_volume,IVV_volume,XLF_volume
"datetime[ns, US/Eastern]",f64,f64,f64,f64,f64
2025-01-02 09:30:00 EST,26119.0,497.0,1.126182e6,73315.0,240736.0
2025-01-02 09:30:30 EST,3574.0,633.0,160721.0,4264.0,87350.0
2025-01-02 09:31:00 EST,6260.0,190.0,94992.0,1387.0,85364.0
2025-01-02 09:31:30 EST,2594.0,0.0,80065.0,5152.0,45702.0
2025-01-02 09:32:00 EST,7492.0,68743.0,68693.0,2036.0,69624.0


### Common Factor Model

#### Abnormal Volume Calulcation

In [48]:
volume_cols = [c for c in df_volume_bucket.columns if c.endswith("_volume")]

# Add time-of-day bucket
df_volume_bucket = df_volume_bucket.with_columns([
    pl.col("ts").dt.date().alias("date"),
    pl.col("ts").dt.time().alias("tod"),
])

# Compute TOD average volume for each 30s bucket
df_tod_avg = (
    df_volume_bucket
    .group_by("tod")
    .agg([
        pl.col(c).mean().alias(c.replace("_volume", "_tod_avg"))
        for c in volume_cols
    ])
    .sort("tod")
)

df_tod_avg.head()

tod,GS_tod_avg,MS_tod_avg,SPY_tod_avg,IVV_tod_avg,XLF_tod_avg
time,f64,f64,f64,f64,f64
09:30:00,53026.275862,27590.551724,551882.698276,84220.025862,373240.931034
09:30:30,12354.922414,8724.327586,164556.534483,8611.836207,110777.37931
09:31:00,7714.931034,14417.827586,183749.474138,10638.913793,137702.362069
09:31:30,5558.387931,15771.431034,132714.068966,7796.948276,119857.767241
09:32:00,6041.396552,23554.758621,131156.284483,8502.017241,135569.275862


In [ ]:
# For each bucket, calculate abnormal volume as log(1 + volume) - log(1 + TOD average) using rolling mean
volume_cols = [c for c in df_volume_bucket.columns if c.endswith("_volume")]

# Add date and time-of-day bucket
df = (
    df_volume_bucket
    .with_columns([
        pl.col("ts").dt.date().alias("date"),
        pl.col("ts").dt.time().alias("tod"),
    ])
    .sort(["tod", "date"])
)

# One row per date/tod should already exist, but this is a safety aggregation
daily_tod = (
    df
    .group_by(["date", "tod"])
    .agg([
        pl.col(c).sum().alias(c)
        for c in volume_cols
    ])
    .sort(["tod", "date"])
)

# Past-only expanding TOD average:
# For each tod bucket, average previous days only.
df_expanding_tod = daily_tod.with_columns([
    (
        pl.col(c).cum_sum().shift(1).over("tod")
        /
        pl.col(c).cum_count().shift(1).over("tod")
    ).alias(c.replace("_volume", "_tod_avg_past"))
    for c in volume_cols
])

# Join past-only TOD averages back to bucket df
tod_avg_cols = [
    c.replace("_volume", "_tod_avg_past")
    for c in volume_cols
]

df_abvol = df.join(
    df_expanding_tod.select(["date", "tod"] + tod_avg_cols),
    on=["date", "tod"],
    how="left"
)

# Calculate leakage-safe abnormal volume
df_abvol = df_abvol.with_columns([
    (
        pl.col(c).log1p()
        - pl.col(c.replace("_volume", "_tod_avg_past")).log1p()
    ).alias(c.replace("_volume", "_abvol"))
    for c in volume_cols
])

# Drop rows where there is no past TOD average, usually first trading day
df_abvol = df_abvol.drop_nulls(tod_avg_cols)

df_abvol.head()

ts,GS_volume,MS_volume,SPY_volume,IVV_volume,XLF_volume,date,tod,GS_tod_avg_past,MS_tod_avg_past,SPY_tod_avg_past,IVV_tod_avg_past,XLF_tod_avg_past,GS_abvol,MS_abvol,SPY_abvol,IVV_abvol,XLF_abvol
"datetime[ns, US/Eastern]",f64,f64,f64,f64,f64,date,time,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2025-01-03 09:30:00 EST,24937.0,100.0,875674.0,55773.0,309530.0,2025-01-03,09:30:00,26119.0,497.0,1.126182e6,73315.0,240736.0,-0.046309,-1.59548,-0.251594,-0.273471,0.251353
2025-01-06 09:30:00 EST,41070.0,3970.0,655496.0,93763.0,284627.0,2025-01-06,09:30:00,25528.0,298.5,1.000928e6,64544.0,275133.0,0.475487,2.584659,-0.42329,0.373418,0.033925
2025-01-07 09:30:00 EST,22449.0,269.0,318134.0,71186.0,280201.0,2025-01-07,09:30:00,30708.666667,1522.333333,885784.0,74283.666667,278297.666667,-0.313287,-1.730234,-1.023998,-0.042594,0.006816
2025-01-08 09:30:00 EST,24462.0,69923.0,319251.0,49822.0,208543.0,2025-01-08,09:30:00,28643.75,1209.0,743871.5,73509.25,278773.5,-0.157808,4.056789,-0.845889,-0.388948,-0.290253
2025-01-10 09:30:00 EST,44625.0,100.0,480172.0,137876.0,397937.0,2025-01-10,09:30:00,27807.4,14951.8,658947.4,68771.8,264727.4,0.472979,-4.997533,-0.316499,0.695554,0.407592


#### PCA for abvol cov

In [59]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

pca_cols = ["MS_abvol", "SPY_abvol", "IVV_abvol", "XLF_abvol"]
rolling_days = 20

# Make sure sorted by date/time
df = df_abvol.sort(["date", "ts"])

# Get unique trading days
dates = df.select("date").unique().sort("date")["date"].to_list()

outputs = []

for i in range(rolling_days, len(dates)):
    fit_days = dates[i - rolling_days:i]   # past 20 trading days
    current_day = dates[i]                 # day we are computing pca1 for

    df_fit = df.filter(pl.col("date").is_in(fit_days))
    df_current = df.filter(pl.col("date") == current_day)

    # Fit scaler only on past data
    X_fit = df_fit.select(pca_cols).drop_nulls().to_numpy()
    X_current = df_current.select(pca_cols).to_numpy()

    if X_fit.shape[0] == 0 or X_current.shape[0] == 0:
        continue

    scaler = StandardScaler()
    X_fit_scaled = scaler.fit_transform(X_fit)

    # Fit PCA only on past data
    pca = PCA(n_components=1)
    pca.fit(X_fit_scaled)

    # Transform current day only
    X_current_scaled = scaler.transform(X_current)
    pca1_current = pca.transform(X_current_scaled).ravel()

    # Attach pca1 to current day
    df_current_out = df_current.with_columns([
        pl.Series("pca1", pca1_current)
    ])

    outputs.append(df_current_out)

df_with_pca = pl.concat(outputs).sort("ts")

In [62]:
df_with_pca.head()

ts,GS_volume,MS_volume,SPY_volume,IVV_volume,XLF_volume,date,tod,GS_tod_avg_past,MS_tod_avg_past,SPY_tod_avg_past,IVV_tod_avg_past,XLF_tod_avg_past,GS_abvol,MS_abvol,SPY_abvol,IVV_abvol,XLF_abvol,pca1
"datetime[ns, US/Eastern]",f64,f64,f64,f64,f64,date,time,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2025-02-04 09:30:00 EST,38381.0,0.0,379735.0,63586.0,270611.0,2025-02-04,09:30:00,79056.190476,44404.809524,671859.619048,84116.571429,410903.238095,-0.722583,-10.701126,-0.570575,-0.279806,-0.417674,-3.117353
2025-02-04 09:30:30 EST,323.0,100.0,131665.0,6127.0,36941.0,2025-02-04,09:30:30,10142.904762,3141.285714,168024.047619,8364.0,140682.285714,-3.443885,-3.437585,-0.243845,-0.311188,-1.337162,-1.292599
2025-02-04 09:31:00 EST,12207.0,608.0,86644.0,8096.0,124577.0,2025-02-04,09:31:00,8887.904762,31354.52381,182250.047619,8872.333333,136891.952381,0.317288,-3.941327,-0.743566,-0.091557,-0.094267,-1.116201
2025-02-04 09:31:30 EST,4470.0,0.0,98802.0,1528.0,72874.0,2025-02-04,09:31:30,6841.857143,9933.619048,109370.571429,7410.190476,102540.857143,-0.425593,-9.203781,-0.101623,-1.578377,-0.341525,-2.700148
2025-02-04 09:32:00 EST,12655.0,170.0,75853.0,3012.0,310492.0,2025-02-04,09:32:00,6895.52381,20720.190476,94112.238095,5170.190476,108137.857143,0.607114,-4.797249,-0.215688,-0.540167,1.054745,-0.514751


#### Train Test split

In [63]:
from datetime import date as Date

train_end = Date.fromisoformat("2025-04-30")

df_model = df_with_pca.filter(pl.col("date").is_not_null())

df_train = df_model.filter(pl.col("date") <= train_end)
df_test = df_model.filter(pl.col("date") > train_end)

#### Improving VWAP strategies: A dynamical volume approach - Jędrzej BIAŁKOWSKI, Serge DAROLLES & Gaëlle LE FOL
$${GS\_abvol}_t = \alpha + \lambda F_t + u_t$$
where $$F_t = pca1_t$$
and $$u_t = {GS\_abvol}_t - \hat{\alpha} - \hat{\lambda}F_t$$
$$u_t \sim ARMA\ or\ SETAR$$

#### 1. ARMA
$$u_t = {GS\_idio\_vol}_t$$
Fit
$$u_t = c + \phi_1 u_{t-1} + ... + \phi_p u_{t-p} + \theta_1 \epsilon_{t-1} + \theta_q \epsilon_{q} + \epsilon_t$$
Predict
$$\hat{u}_{t+1} = c + \sum_{j=1}^{p} \hat{\phi}_j u_{t+1-j} + \sum_{j=1}^{q} \hat{\theta}_j \hat{\epsilon}_{t+1-j}$$
Reconstruct
$$\hat{GS\_idio\_vol}_{t+1} = \hat{\alpha} + \hat{\lambda}F_t + \hat{u}_{t+1}$$